In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!find /content/drive/MyDrive -maxdepth 4 -type f -name "rdd2022_yolo.zip"

/content/drive/MyDrive/NEXORA/RoadDamageProject/rdd2022_yolo.zip


In [4]:
!unzip -q "/content/drive/MyDrive/NEXORA/RoadDamageProject/rdd2022_yolo.zip" -d /content/

In [5]:
!find /content/rdd2022_yolo -maxdepth 2 -type d | sort

/content/rdd2022_yolo
/content/rdd2022_yolo/test
/content/rdd2022_yolo/test/images
/content/rdd2022_yolo/test/labels
/content/rdd2022_yolo/train
/content/rdd2022_yolo/train/images
/content/rdd2022_yolo/train/labels
/content/rdd2022_yolo/val
/content/rdd2022_yolo/val/images
/content/rdd2022_yolo/val/labels


In [6]:
from pathlib import Path

base = Path("/content/rdd2022_yolo")

for split in ["train", "val", "test"]:
    images = list((base / split / "images").glob("*"))
    labels = list((base / split / "labels").glob("*.txt"))

    print(f"{split:5} | images: {len(images):5} | labels: {len(labels):5}")


train | images: 30707 | labels: 30707
val   | images:  3833 | labels:  3833
test  | images:  3845 | labels:  3845


In [7]:
!nvidia-smi


Sun Aug 30 05:53:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.2 MB/s eta 0:00:00


In [9]:
from ultralytics import YOLO
import ultralytics

print("Ultralytics:", ultralytics.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics: 8.4.135


In [10]:
%%writefile /content/rdd2022_yolo/data.yaml

path: /content/rdd2022_yolo

train: train/images
val: val/images
test: test/images

names:
  0: D00
  1: D10
  2: D20
  3: D40

Overwriting /content/rdd2022_yolo/data.yaml


In [11]:
!cat /content/rdd2022_yolo/data.yaml


path: /content/rdd2022_yolo

train: train/images
val: val/images
test: test/images

names:
  0: D00
  1: D10
  2: D20
  3: D40


In [12]:
from pathlib import Path

base = Path("/content/rdd2022_yolo")
valid_classes = {0, 1, 2, 3}

bad_class_ids = []
bad_boxes = []
empty_labels = 0
total_objects = 0

for split in ["train", "val", "test"]:
    for label_file in (base / split / "labels").glob("*.txt"):
        text = label_file.read_text().strip()

        if not text:
            empty_labels += 1
            continue

        for line in text.splitlines():
            parts = line.split()

            if len(parts) != 5:
                bad_boxes.append((split, label_file.name, line))
                continue

            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:])

            total_objects += 1

            if cls not in valid_classes:
                bad_class_ids.append((split, label_file.name, cls))

            if not (
                0 <= x <= 1
                and 0 <= y <= 1
                and 0 < w <= 1
                and 0 < h <= 1
            ):
                bad_boxes.append((split, label_file.name, line))

print("Total objects:", total_objects)
print("Empty label files:", empty_labels)
print("Invalid class IDs:", len(bad_class_ids))
print("Invalid YOLO boxes:", len(bad_boxes))

Total objects: 55006
Empty label files: 14618
Invalid class IDs: 0
Invalid YOLO boxes: 0


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data="/content/rdd2022_yolo/data.yaml",
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    patience=10,
    project="/content/drive/MyDrive/NEXORA/RoadDamageProject/runs",
    name="yolo11s_baseline",
    save=True
)

Ultralytics 8.4.135 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/rdd2022_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_baseline, nbs=64, nms=